# 🔍 Multimodal Crime / Incident Report Analyzer
## Demo Notebook — 3 Modalities: PDF · Audio · Text NLP
**COMP 4XXX — Final Assignment | Group of 2**

This notebook walks through the full AI pipeline:
1. **Modality 1 — PDF Processor**: pdfplumber text extraction + Tesseract OCR for scanned PDFs
2. **Modality 2 — Audio Transcriber**: OpenAI Whisper speech-to-text + GPT-4o extraction
3. **Modality 3 — Text NLP Analyzer**: spaCy NER + DistilBERT sentiment + keyword classification
4. **Integrator**: Cross-modality fusion → one structured row per real-world incident
5. **Output**: CSV + JSON dataset + Streamlit dashboard

## Cell 1 — Install Dependencies

In [ ]:
# Run this cell first to install all required packages
!pip install -q openai pdfplumber pdf2image pytesseract Pillow openai-whisper \
    spacy transformers torch pandas python-dotenv streamlit plotly
!python -m spacy download en_core_web_sm
# Linux OCR system deps (Colab / Ubuntu):
!apt-get install -qq tesseract-ocr poppler-utils
print('✅ All dependencies installed!')

## Cell 2 — Configure API Key

In [ ]:
import os
# Option A: set directly (Colab)
os.environ['OPENAI_API_KEY'] = 'your-api-key-here'

# Option B: load from .env file
# from dotenv import load_dotenv; load_dotenv()

print('API key set:', bool(os.environ.get('OPENAI_API_KEY')))

## Cell 3 — Modality 1: PDF Processor
Processes police incident reports (text-based PDFs and scanned/image PDFs with OCR).

In [ ]:
import sys, json
sys.path.insert(0, '..')
from src.pdf_processor.processor import process_pdf

# Process a sample police report (text file simulates PDF content for demo)
pdf_result = process_pdf('../data/samples/pdf_samples/HPD_report_burglary.txt')

print('=== PDF Module Output ===')
for key, val in pdf_result.items():
    if key not in ('raw_text',):
        print(f'  {key:25s}: {val}')

## Cell 4 — Modality 2: Audio Transcriber
Transcribes audio files using OpenAI Whisper, then extracts structured fields with GPT-4o.

In [ ]:
from src.audio_transcriber.transcriber import extract_fields_from_transcript

# Demo: use a pre-written transcript (place real .mp3 file and call process_audio() for real run)
sample_transcript = """
DISPATCHER: 911, what's your emergency?
CALLER: Yes hello — there's a fight outside Club Luxe on Westheimer and Montrose.
Two guys, one of them hit the other with a glass bottle. There's blood everywhere. Please hurry.
DISPATCHER: Are you safe? Can you describe the suspects?
CALLER: Yes I'm across the street. One has a red jacket, about 6 feet tall. The other has
a white shirt. The victim is on the ground.
DISPATCHER: Units are 2 minutes away.
"""

audio_result = extract_fields_from_transcript(sample_transcript)
audio_result['source_modality'] = 'audio'
audio_result['source_file'] = '911_call_demo.mp3'
audio_result['transcript'] = sample_transcript

print('=== Audio Module Output ===')
for key, val in audio_result.items():
    if key != 'transcript':
        print(f'  {key:25s}: {val}')

## Cell 5 — Modality 3: Text NLP Analyzer
Runs spaCy NER + DistilBERT sentiment + keyword topic classification + GPT-4o extraction.

In [ ]:
from src.text_nlp.analyzer import process_text, run_ner, run_sentiment, classify_topic

sample_text = """
On the night of November 20th at around 11:45 PM, I witnessed two men attacking
a woman near the bus stop on Westheimer Road outside Club Luxe. One man was wearing
a red jacket, about 6 feet tall. The other was shorter, around 5 feet 8, in a white hoodie.
The woman was screaming for help. They grabbed her black leather purse and ran toward
the parking garage on Montrose. I called 911 immediately.
My name is Robert Chen, 4200 Main St, Houston TX.
"""

print('--- spaCy NER ---')
ner = run_ner(sample_text)
for k, v in ner.items():
    if v: print(f'  {k}: {v}')

print('\n--- DistilBERT Sentiment ---')
sent = run_sentiment(sample_text)
print(f'  label={sent["label"]}, score={sent["score"]:.3f}, tone={sent["tone"]}')

print('\n--- Keyword Topic Classification ---')
topic = classify_topic(sample_text)
print(f'  primary: {topic["primary_topic"]} (score={topic["keyword_score"]})')
print(f'  all scores: {topic["all_scores"]}')

## Cell 6 — Integration: Fuse all modalities → one structured dataset

In [ ]:
import sys; sys.path.insert(0, '..')
from src.integrator.merge import run_integration

# Full demo records (3 incidents, 7 records)
sys.path.insert(0, '..')
exec(open('../run_pipeline.py').read().split('def parse_args')[0])

result = run_integration(DEMO_RECORDS, output_dir='../data/outputs')

df = result['dataframe']
print(f'\nFused {result["record_count"]} incident rows from {len(DEMO_RECORDS)} modality records')
df[['incident_id','date','incident_type','combined_severity',
    'sources_present','combined_confidence']].head(10)

## Cell 7 — View the structured dataset

In [ ]:
import pandas as pd
df = pd.read_csv('../data/outputs/structured_incidents_latest.csv')
print(f'Dataset shape: {df.shape}')
print(f'Columns ({len(df.columns)}): {list(df.columns)}')
df.head()

## Cell 8 — Launch the dashboard
Run the Streamlit dashboard to explore the dataset interactively.

In [ ]:
# In a terminal (not inside Jupyter) run:
#   streamlit run dashboard/app.py
# Or from this notebook:
print('To launch the dashboard, run in a terminal:')
print('  cd crime_analyzer')
print('  streamlit run dashboard/app.py')